# Multi-turn GSM8K async RL with a feedback loop — walkthrough

An educational, **runnable** tour of the [`multi_turn_message_in`](./train.py) recipe, retargeted to
**GLM-5.1** (`accounts/fireworks/models/glm-5p1`).

We train a math agent on GSM8K. The model is asked a problem and must put its final
answer in `\boxed{...}`. If the boxed answer is **wrong**, the rollout appends a fixed
user-feedback message and lets the model **retry once**. The whole trajectory — prompt +
attempt 1 + feedback + attempt 2 — is packed into a single, loss-masked training sample.

```
         dataset row {messages, answer}
                    |
                    v
   +--------------------------------------+
   |   rollout_fn  (YOU write this)       |
   |                                      |
   |   turn 1: sample -> score boxed ans  |
   |        correct? --yes--> done        |
   |           | no                       |
   |           v                          |
   |   append RETRY_PROMPT                 |
   |   turn 2: sample -> score boxed ans  |
   |                                      |
   |   pack tokens/logprobs/loss_mask     |
   |   -> RolloutRun(reward = last turn)  |
   +--------------------------------------+
                    |
                    v
   recipe owns: fan-out x completions_per_prompt, GRPO advantage,
   reference KL, PPO inner loop, weight sync, checkpoint + promote
```

**The split that makes this recipe small:** you only write the rollout function. Everything
between the rollout and the optimizer step is owned by `recipes/async_rl_loop.py::main`.

### GLM-5.1 specifics
GLM-5.1 is a large Mixture-of-Experts model, so this notebook differs from the stock
Qwen-1.5B example in three ways:
- **LoRA** (`lora_rank > 0`) — full-parameter RL on a model this size is impractical.
- **An explicit training shape** — GLM-5.1 has no auto-selectable default, so we pin the
  validated LoRA shape `glm-5p1-200k-lora` (see §7 for why leaving it unset fails with a
  *"Only superuser can skip validations"* 400).
- **TIS stays on** (the recipe default) to correct the train/inference numerics gap, which
  is wider for the quantized (MXFP8) MoE deployment GLM-5.1 serves on. R3 routing replay
  (`routing_matrices`) is an optional advanced lever; not required to get a run going.

> ⚠️ Sections 1–7 run **locally** (CPU, just a tokenizer download — no GPUs, no cost).
> Section 8 (**GO LIVE**) provisions a real Fireworks trainer + inference deployment and
> consumes GPU quota. Run it deliberately.

## 1. Environment

We need the repo root on `sys.path` (so `import training...` works) and — for the live run —
`FIREWORKS_API_KEY` in the environment. The local sections don't need the key.

In [1]:
import os, sys, json, pathlib
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
# This notebook lives in training/examples/rl/multi_turn_message_in/.
# Walk up to the repo root (the dir that *contains* the `training` package).
here = pathlib.Path.cwd()
repo_root = next(p for p in [here, *here.parents] if (p / "training" / "__init__.py").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("repo root:", repo_root)

print("FIREWORKS_API_KEY set:", bool(os.environ.get("FIREWORKS_API_KEY")))

# WandB: WANDB_ENTITY is REQUIRED for metric logging (the recipe disables
# wandb if it is empty); WANDB_API_KEY syncs to the dashboard (else offline).
print("WANDB_ENTITY    :", os.environ.get("WANDB_ENTITY") or "(unset -> NO metrics)")
print("WANDB_API_KEY   :", "set" if os.environ.get("WANDB_API_KEY") else "(unset -> offline only)")

repo root: /Users/sinan/cookbook
FIREWORKS_API_KEY set: True
WANDB_ENTITY    : fireworks-devrel
WANDB_API_KEY   : set


In [2]:
# Model + tokenizer for GLM-5.1. The base model is a Fireworks account model id;
# the tokenizer is the matching HuggingFace repo (its chat template is what the
# message assembler renders with).
BASE_MODEL = "accounts/fireworks/models/glm-5p1"
TOKENIZER_MODEL = "zai-org/GLM-5.1"
MAX_TURNS = 2          # AReaL default: 1 first attempt + 1 retry
COMPLETIONS_PER_PROMPT = 4   # GRPO needs >= 2 so the group advantage is defined
print(BASE_MODEL, "|", TOKENIZER_MODEL)

accounts/fireworks/models/glm-5p1 | zai-org/GLM-5.1


## 2. Data

Each dataset row is `{"messages": [...], "answer": "..."}`. The `answer` is the *full* GSM8K
chain-of-thought string ending in `#### N` — the reward function parses `N` out of it.
The prompt carries a suffix instructing the model to box its final answer.

We pull a few real rows from HuggingFace `openai/gsm8k` (the same source as
[`prepare_data.py`](./prepare_data.py)).

In [3]:
from datasets import load_dataset

PROMPT_SUFFIX = "\nPlease put your final answer within \\boxed{}."

ds = load_dataset("openai/gsm8k", "main", split="train")

def to_row(idx, item):
    return {
        "id": f"gsm8k-train-{idx}",
        "messages": [{"role": "user", "content": item["question"] + PROMPT_SUFFIX}],
        "answer": item["answer"],
    }

rows = [to_row(i, ds[i]) for i in range(16)]   # small slice for the walkthrough
print(json.dumps(rows[0], indent=2)[:900])

/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "id": "gsm8k-train-0",
  "messages": [
    {
      "role": "user",
      "content": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nPlease put your final answer within \\boxed{}."
    }
  ],
  "answer": "Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72"
}


## 3. The reward

[`reward.py`](./reward.py)'s `gsm8k_reward(completion, answer)` returns `1.0` iff the
*last* `\boxed{...}` in the completion matches the ground-truth `#### N`. It tries a cheap
numeric match first, then falls back to `math_verify` for fractions/surds. It never raises
— anything unparsable is just `0.0`. Let's watch it on three cases.

In [4]:
from training.examples.rl.multi_turn_message_in.reward import gsm8k_reward

gt = "She has 3 + 5 = 8 apples.\n#### 8"   # ground-truth answer string
cases = {
    "correct":     "The total is \\boxed{8}.",
    "wrong":       "I think it is \\boxed{7}.",
    "unparsable":  "The answer is eight.",          # no \boxed{} -> 0.0
}
for name, completion in cases.items():
    print(f"{name:12s} -> reward = {gsm8k_reward(completion, gt)}")

correct      -> reward = 1.0
wrong        -> reward = 0.0
unparsable   -> reward = 0.0


## 4. The heart: token-level loss masking

This is the one idea worth slowing down for. A multi-turn trajectory is flattened into
**three parallel lists**:

| list | meaning |
|---|---|
| `tokens` | every token in the trajectory (prompt + all turns) |
| `logprobs` | inference logprob per token (`0.0` on non-generated positions) |
| `loss_mask` | **`1` on assistant-generated tokens, `0` everywhere else** |

The trainer multiplies the policy-gradient loss by `loss_mask`, so the original prompt and
the user-feedback bridge between turns contribute **zero gradient** — we only train on what
the model actually generated, across *both* turns.

`MessageTrajectoryAssembler` builds these lists. Crucially it is **TITO** (token-in,
token-out): it keeps the engine's exact completion token IDs for assistant turns and only
tokenizes the appended user/feedback text — so there's never a re-tokenization mismatch.

Below we simulate a *wrong-then-right* trajectory **locally** (no deployment): we fake two
assistant completions by tokenizing text with the real GLM-5.1 tokenizer, then inspect the
mask. This downloads only the tokenizer, not the model.

In [5]:
from transformers import AutoTokenizer
from training.utils.rl.rollout import MessageTrajectoryAssembler, TITOTokenizer
from training.examples.rl.multi_turn_message_in.rollout import RETRY_PROMPT

tok = AutoTokenizer.from_pretrained(TOKENIZER_MODEL, trust_remote_code=True)
assembler = MessageTrajectoryAssembler(TITOTokenizer(tok))

def fake_completion(text):
    """Stand in for an engine sample: real token ids + placeholder logprobs."""
    ids = tok.encode(text, add_special_tokens=False)
    return ids, [0.0] * len(ids)

messages = list(rows[0]["messages"])

# --- turn 1: model answers WRONG ---
p1 = assembler.prepare_next_input(messages)
a1_text = "Let me add them: 3 + 5 = 9. \\boxed{9}"
a1_ids, a1_lp = fake_completion(a1_text)
a1_msg = {"role": "assistant", "content": a1_text}
assembler.add_assistant_response(
    request_messages=messages, assistant_message=a1_msg,
    prompt_token_ids=p1, completion_token_ids=a1_ids, completion_logprobs=a1_lp,
)

# wrong -> append the fixed feedback message and retry
messages2 = messages + [a1_msg, {"role": "user", "content": RETRY_PROMPT}]

# --- turn 2: model answers RIGHT ---
p2 = assembler.prepare_next_input(messages2)
a2_text = "On reflection, 3 + 5 = 8. \\boxed{8}"
a2_ids, a2_lp = fake_completion(a2_text)
assembler.add_assistant_response(
    request_messages=messages2, assistant_message={"role": "assistant", "content": a2_text},
    prompt_token_ids=p2, completion_token_ids=a2_ids, completion_logprobs=a2_lp,
)

tokens, logprobs, loss_mask = assembler.trajectory.to_flat()
n_train = sum(loss_mask)
print(f"total tokens : {len(tokens)}")
print(f"trained (mask=1): {n_train}   ({100*n_train/len(tokens):.0f}% of trajectory)")
print(f"masked  (mask=0): {len(tokens) - n_train}  <- prompt + feedback bridge")

Builtin loss patch skipped: ForwardBackwardInput.model_fields not found


total tokens : 131
trained (mask=1): 36   (27% of trajectory)
masked  (mask=0): 95  <- prompt + feedback bridge


In [6]:
# Visualize the mask as a 1/0 strip and confirm the two trained spans (the two
# assistant turns) line up with the boxed answers.
strip = "".join(str(m) for m in loss_mask)
print("loss_mask (1=trained assistant token, 0=prompt/feedback):\n")
for i in range(0, len(strip), 80):
    print(strip[i:i+80])

# Spans of consecutive 1s == assistant turns.
spans, start = [], None
for i, m in enumerate(loss_mask + [0]):
    if m and start is None: start = i
    elif not m and start is not None: spans.append((start, i)); start = None
print("\ntrained spans:", spans)
for s, e in spans:
    print(f"  [{s}:{e}] ->", repr(tok.decode(tokens[s:e])[:70]))

loss_mask (1=trained assistant token, 0=prompt/feedback):

00000000000000000000000000000000000000000000000000001111111111111111111000000000
000000000000000000000000000000000011111111111111111

trained spans: [(52, 71), (114, 131)]
  [52:71] -> 'Let me add them: 3 + 5 = 9. \\boxed{9}'
  [114:131] -> 'On reflection, 3 + 5 = 8. \\boxed{8}'


## 5. The rollout function

[`rollout.py`](./rollout.py)'s `make_rollout_fn(setup) -> rollout_fn` is the only thing you
*have* to write. Annotated, the inner loop is exactly the dance we just did by hand, but
with **real samples** from the inference deployment instead of `fake_completion`:

```python
async def rollout_fn(sample_prompt: dict) -> RolloutRun | None:
    messages = sample_prompt["messages"]
    answer   = sample_prompt["answer"]
    assembler = MessageTrajectoryAssembler(TITOTokenizer(tokenizer))
    current_messages, last_reward = messages, 0.0

    for turn in range(max_turns):
        prompt_tokens = assembler.prepare_next_input(current_messages)
        completion = (await sampler.sample_with_prompt_tokens(prompt_tokens, n=1, **kw))[0]
        out_tokens = completion.full_tokens[completion.prompt_len:]   # generated only
        assembler.add_assistant_response(... out_tokens ..., out_logprobs ...)

        last_reward = gsm8k_reward(completion.text, answer)
        if last_reward >= 1.0:           # got it right -> stop early
            break
        if turn + 1 < max_turns:         # wrong -> append feedback, loop
            current_messages = current_messages + [assistant_msg,
                                                    {"role": "user", "content": RETRY_PROMPT}]

    tokens, logprobs, loss_mask = assembler.trajectory.to_flat()
    return RolloutRun(segments=[RolloutSample(tokens, logprobs, loss_mask, reward=last_reward)])
```

**Why the scalar reward is just the last turn:** the recipe fans each row out to
`completions_per_prompt` trajectories and compares them with GRPO. A "wrong-then-right"
trajectory (reward 1.0) naturally beats "wrong-then-wrong" (0.0) within the group — no
explicit per-turn discount needed.

In [7]:
# Confirm the factory imports and is the callable the recipe expects.
from training.examples.rl.multi_turn_message_in.rollout import make_rollout_fn
import inspect
print("make_rollout_fn signature:", inspect.signature(make_rollout_fn))
print("RETRY_PROMPT:\n ", RETRY_PROMPT)

make_rollout_fn signature: (setup: "'RolloutSetup'") -> "'RolloutFn'"
RETRY_PROMPT:
  Your answer is either wrong or not parsable to the reward function. You may misunderstand the original question. Please carefully read the original question, check the previous errors, and try to answer it again.


## 6. GRPO advantage (why `completions_per_prompt >= 2`)

The recipe computes the advantage by **z-scoring rewards within each prompt group**
(GRPO). With a single completion the standard deviation is undefined, so the group is
dropped — which is why `main()` rejects `completions_per_prompt < 2`. Toy demo:

In [8]:
import torch

def grpo_advantage(rewards):
    r = torch.tensor(rewards, dtype=torch.float32)
    return (r - r.mean()) / (r.std() + 1e-6)

# 4 trajectories for one GSM8K problem: two solved it, two didn't.
group_rewards = [1.0, 1.0, 0.0, 0.0]
print("rewards   :", group_rewards)
print("advantage :", grpo_advantage(group_rewards).tolist())

# A 'constant' group (all same reward) has ~zero advantage -> a no-op optimizer step.
# train.py's --filter-constant-reward drops these. This is dynamic_filter_fn:
print("\nconstant group [1,1,1,1] advantage:", grpo_advantage([1.0, 1.0, 1.0, 1.0]).tolist())
dynamic_filter_fn = lambda pg: len(set(pg.rewards)) > 1   # keep only groups with signal

rewards   : [1.0, 1.0, 0.0, 0.0]
advantage : [0.8660238981246948, 0.8660238981246948, -0.8660238981246948, -0.8660238981246948]

constant group [1,1,1,1] advantage: [0.0, 0.0, 0.0, 0.0]


## 7. Build the `Config` for the GLM-5.1 run

Everything below feeds `recipes/async_rl_loop.main`. Knobs chosen for a small, cheap-ish
first run; scale `max_rows`, `prompt_groups_per_step`, and `replica_count` up once it works.

**Training shape (required for GLM-5.1).** Small models auto-select a shape when
`training_shape_id` is unset, but GLM-5.1 has no auto-default — and an unset shape makes the
SDK fall back to `skipValidations=true`, which a normal (non-superuser) account is not
allowed to use (the backend returns *"Only superuser can skip validations"* → HTTP 400). So
we pin the validated GLM-5.1 LoRA shape `accounts/fireworks/trainingShapes/glm-5p1-200k-lora`
(its paired deployment shape is the quantized `glm-5p1-rft-b300-mxfp8-w8-p1` — which is why
TIS matters). We still don't set `region` or any manual accelerator fields; the backend
places the colocated deployment.

In [9]:
import time
from training.recipes.async_rl_loop import Config
from training.utils import DeployConfig, TrainerConfig, WandBConfig

OUTPUT_MODEL_ID = None   # e.g. "accounts/<your-acct>/models/gsm8k-mt-glm5p1"; None = don't promote

# GLM-5.1 LoRA trainer shape. Required: GLM-5.1 has no auto-selectable default, and
# leaving training_shape_id unset makes the SDK send skipValidations=true, which
# non-superuser accounts can't use ("Only superuser can skip validations" -> HTTP 400).
# This shape is LORA_TRAINER mode, 8xB300 / 1 node, 200K ctx, paired with the RFT
# deployment shape glm-5p1-rft-b300-mxfp8-w8-p1. Pass the BARE path (no /versions/...);
# the platform picks the latest validated version.
TRAINING_SHAPE_ID = "accounts/fireworks/trainingShapes/glm-5p1-200k-lora"

cfg = Config(
    log_path="./gsm8k_mt_glm5p1_logs",
    base_model=BASE_MODEL,
    learning_rate=1.7e-5,
    kl_beta=0.0,                       # >0 spins up a reference model for KL
    lora_rank=16,                      # LoRA RL on the MoE (full-param is impractical here)
    completions_per_prompt=COMPLETIONS_PER_PROMPT,
    max_completion_tokens=1024,
    temperature=1.0,
    epochs=1,
    max_rows=len(rows),
    prompt_groups_per_step=2,          # rollout batch size, in prompt groups
    max_head_offpolicy_versions=0,     # strict on-policy submission budget
    ppo_n_minibatches=1,
    output_model_id=OUTPUT_MODEL_ID,
    trainer=TrainerConfig(training_shape_id=TRAINING_SHAPE_ID),
    deployment=DeployConfig(tokenizer_model=TOKENIZER_MODEL, replica_count=1),
    wandb=WandBConfig(
        entity=os.environ.get("WANDB_ENTITY", ""),
        project=os.environ.get("WANDB_PROJECT", "gsm8k-mt-glm5p1"),
        run_name=f"gsm8k-mt-glm5p1-{int(time.time()) % 100000}",
    ),
)
print("Config ready. base =", cfg.base_model, "| lora_rank =", cfg.lora_rank,
      "| shape =", cfg.trainer.training_shape_id, "| rows =", cfg.max_rows)

Config ready. base = accounts/fireworks/models/glm-5p1 | lora_rank = 16 | shape = accounts/fireworks/trainingShapes/glm-5p1-200k-lora | rows = 16


## 8. ⚠️ GO LIVE — provision infra and train

**This cell costs money and GPU quota.** Calling `main()` will:
1. Provision a Fireworks **training job** for `glm-5p1` (LoRA) and a colocated
   **inference deployment** for sampling.
2. Run the async RL loop: fan out rollouts → GRPO advantage → PPO step → weight-sync the
   updated LoRA into the sampler → repeat.
3. On clean exit, save a checkpoint (and promote to `OUTPUT_MODEL_ID` if you set one).

Requires `FIREWORKS_API_KEY`. The recipe cleans up the trainer/deployment on exit
(`cfg.cleanup_on_exit=True`). Watch progress in the logs and in WandB.

> Tip: for a first smoke run, keep `max_rows` and `prompt_groups_per_step` small. If a step
> never trains, check that rollout groups aren't all constant-reward (section 6).

**WandB metrics.** To see tokens/sec, `train/inference_kld`, entropy, grad_norm and the
reward curve on a dashboard, set `WANDB_ENTITY` and `WANDB_API_KEY` in `training/.env`
(see the env-check cell in section 1). Without `WANDB_ENTITY` the recipe logs **nothing**;
with the entity but no API key it logs **offline** (local `wandb/` dir). The Config cell
below already reads these via `load_dotenv()` — just populate `.env` and re-run from section 1.

In [10]:
assert os.environ.get("FIREWORKS_API_KEY"), "Set FIREWORKS_API_KEY before going live."

# Jupyter already runs an asyncio event loop, but `main()` calls `asyncio.run(...)`
# internally, which Python forbids from inside a running loop. nest_asyncio makes
# asyncio.run re-entrant so the recipe runs unmodified. (A worker thread won't work:
# main() installs SIGTERM/SIGINT handlers via signal.signal, which only works on the
# main thread.)
import nest_asyncio  # pip install nest_asyncio if missing
nest_asyncio.apply()

from training.recipes.async_rl_loop import main

result = main(
    cfg,
    rollout_fn_factory=make_rollout_fn,
    rows=rows,
    rollout_extras={"max_turns": MAX_TURNS},
    dynamic_filter_fn=dynamic_filter_fn,   # drop constant-reward groups (section 6)
)
print(result)

async_rl_loop is EXPERIMENTAL and under active development; the Config / RolloutSetup API may change. See skills/dev/references/rl/async-rl.md.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sinan-fireworks (fireworks-devrel) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


ctx/completions_per_prompt,▁▁
ctx/max_completion_tokens,▁▁
ctx/max_head_offpolicy_versions,▁▁
ctx/ppo_n_minibatches,▁▁
ctx/prompt_groups_per_step,▁▁
ctx/seed,▁▁
ctx/shuffle,▁▁
ctx/temperature,▁▁
ctx/weight_sync_interval,▁▁
perf/fwd_bwd_time,█▁
+52,...


{'steps': 2, 'policy_job_id': 'training-api-service-ac1899b4', 'reference_job_id': None, 'deployment_id': 'glm-5p1-1781702848'}


## 9. After the run

`main()` returns `{steps, policy_job_id, reference_job_id, deployment_id}`. If you set
`OUTPUT_MODEL_ID`, the final LoRA checkpoint was promoted there and is ready to deploy.

**Where to go next:**
- **Scale**: raise `max_rows`, `prompt_groups_per_step`, and `deployment.replica_count`
  (more replicas fan out sampling) once the smoke run is healthy.
- **Watch policy drift**: `train/inference_kld` / `train/inference_diff` in WandB. For a
  quantized MoE these matter; if they blow up, keep TIS on and consider R3 routing replay
  (`RolloutSample.routing_matrices`).
- **Synchronous baseline**: set `synchronous_training=True` to remove rollout/train overlap
  and measure what the async overlap buys you.
- **Reference KL**: set `kl_beta > 0` to pull toward the base model (spins up a reference).
- The production entrypoint for the same recipe is [`train.py`](./train.py) /
  [`run.sh`](./run.sh); see [`skills/dev/references/rl/async-rl.md`](../../../../skills/dev/references/rl/async-rl.md)
  for the full `Config` contract and gate semantics.